# 영화 리뷰 워드 임베딩 (Word2Vec, FastText)
- gensim 라이브러리 사용 : pip install gensim
    - Word2Vec : models.Word2Vec
    - FastText : models.FastText

## 1. 데이터 준비
* 토큰화가 잘 되어 있는 filtered 데이터 사용

In [1]:
import pandas as pd
data_filename = './data/Korean_movie_reviews_2016_filtered.csv'
data_df = pd.read_csv(data_filename)
data_df.head()

AttributeError: module 'pyarrow' has no attribute '__version__'

In [ ]:
data_df.info

<bound method DataFrame.info of                                                    review  rate
0                                아니 딴 그렇 비 비탄 총 대체 왜 들 온겨     7
1           진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임     1
2       역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 ...    10
3                                          온종일 불편한 피 범벅 일     6
4         답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐     1
...                                                   ...   ...
788184                                           지금 만나러 갑    10
788185                                       와 진짜 완전 재미있었    10
788186                     무조건 아이맥스 보세 영 상미 말 안되 마블 또 일 냈    10
788187  아이맥스 시사회 다녀왔 꼭 로 보시 추천 드릴 초반 전개 너무 빠른 빼 괜찮 배우 ...    10
788188                                          최고 베니 사랑해    10

[788189 rows x 2 columns]>

In [ ]:
# 결측치 제거 (rate에 결측치)
# 결측치 제거하지 않으면 review 데이터 추출 시 float으로 타입 변환됨   
data_df.dropna(inplace=True)

In [ ]:
data_df.info()

<class 'pandas.DataFrame'>
Index: 785448 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   review  785448 non-null  str  
 1   rate    785448 non-null  int64
dtypes: int64(1), str(1)
memory usage: 72.3 MB


In [ ]:
# review만 모아서 review별 토큰 리스트로 변환 : review가 Object 타입이므로 str로 변환 후 split
review_list = list(data_df.review)
review_list[:5]

['아니 딴 그렇 비 비탄 총 대체 왜 들 온겨',
 '진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임',
 '역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 실사 했 사람 거르려 그냥 봤 정말 흠잡 없는 최고 좀비 영화 잔인 거 싫어하지 참고 볼 만하 로미 인물 왜 그런 모르',
 '온종일 불편한 피 범벅 일',
 '답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐']

In [ ]:
type(review_list[0])

str

In [ ]:
token_list = [review.split() for review in review_list]
print(token_list[:2])


#token_list = [ ]
#for review in review_list:
    #if review:
        #token_list.append(review.split())

[['아니', '딴', '그렇', '비', '비탄', '총', '대체', '왜', '들', '온겨'], ['진심', '쓰레기', '영화', '만들', '무서', '알', '쫄아', '틀었', '이건', '뭐', '웃', '거리', '없는', '쓰레기', '영화', '임']]


## 1. Word2Vec 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/word2vec.html

### Skipgram, negative=10 인 경우

In [ ]:
# Word2Vec 모델 생성 및 학습 : window=3, min_count=3
from gensim.models import Word2Vec
model_sg_n10 = Word2Vec(sentences=token_list, vector_size=100, window=3, min_count=1, sg=1, negative=10)

NameError: name 'token_list' is not defined

In [ ]:
# 단어의 임베딩 벡터 확인
model_sg_n10.wv['이정재']

In [ ]:
# 단어의 임베딩 벡터 차원 확인
len(model_sg_n10.wv['이정재'])

In [ ]:
# 두 단어 간 유사도 확인
model_sg_n10.wv
wv.similarity['이정재', '정우성']

In [ ]:
# 특정 단어와 유사한 단어 추출
print([word for word, _ in wv.most_silmilar('재밌', topn=50)])

### Skipgram, negative=5 인 경우

In [ ]:
# 모델 생성
model_sg_n10 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=5)

In [ ]:
# 특어 단어와 유사한 단어 추출 : 이정재
wv = model_sg_n10.wv
wv.most_similar('이정재', topn=20)

In [ ]:
# 특어 단어와 유사한 단어 추출 : 재밌
print([word for word, _ in wv.most_similar('재밌', topn=50)])

### CBOW, negative=10 인 경우

### CBOW, negative=5 인 경우

### OOV(Out of Vocabulary) 문제

In [ ]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in model_sg_n10.wv.key_to_index

NameError: name 'model_sg_n10' is not defined

In [ ]:
# corpus에 없는 단어의 임베딩 벡터 확인 
model_sg_n10.wv['우주평화']

NameError: name 'model_sg_n10' is not defined

## 2. FastText 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/fasttext.html

In [ ]:
# FastText 모델 생성 및 학습
# window=3, min_count=3, min_n=2, max_n=2
from gensim.models import FastText
model = FastText(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=10, min_n=2, max_n=2)
wv = model.wv

In [ ]:
# 특정 단어와 유사한 단어 추출 : 이정재
wv.most_similar('이정재', topn=20)

In [ ]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in wv.key_to_index

In [ ]:
# corpus에 없는 단어의 임베딩 벡터 확인 
wv['우주평화']

In [ ]:
# corpus에 없는 단어와 유사한 단어추출 
wv.most_similar('우주평화', topn=20)